# Cross-Lingual Transfer for Code-Mixed Conversation Summarization

## Research Question

Can fine-tuning on code-mixed conversation summarization in one language pair enable zero-shot transfer to unseen language pairs?

## Experimental Setup

- **Training**: CS-SUM dataset (Chinese-English code-mixed conversations, 2584 samples)
- **In-domain test**: CS-SUM test set (325 samples)
- **Zero-shot test**: GupShup (Hindi-English, 501 samples)
- **Task**: Generate English summaries from code-mixed conversations
- **Model**: Llama-3.2-3B-Instruct with LoRA fine-tuning

## Method: Code-Switching Curriculum Learning (CSCL)

We propose training in three phases ordered by Code-Mixing Index (CMI):

1. **Phase 1 (Easy)**: Low CMI samples - mostly monolingual with few switches
2. **Phase 2 (Medium)**: Medium CMI samples - moderate code-switching
3. **Phase 3 (Hard)**: High CMI samples - heavy code-mixing

## Key Findings

1. Fine-tuned 3B model outperforms GPT-4.1 on both datasets
2. Zero-shot transfer works: Hindi-English performance comparable to in-domain
3. Curriculum learning trades in-domain performance for better cross-lingual generalization

---

## Section 1: Environment Setup

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU required for training"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU: NVIDIA A100-SXM4-80GB
Memory: 85.2 GB


In [ ]:
%%capture
!pip install transformers==4.44.0 accelerate==0.33.0 peft==0.12.0
!pip install datasets==2.20.0 evaluate==0.4.2 rouge-score==0.1.2
!pip install bert-score==0.3.13 nltk==3.8.1 sacrebleu==2.4.2
!pip install langdetect sentencepiece scipy
!pip install openai

In [ ]:
import os
import json
import random
import gc
import warnings
from typing import Dict, List
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, TaskType
import evaluate
import nltk

warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

PyTorch: 2.9.0+cu126
CUDA: 12.6


---

## Section 2: Authentication

In [ ]:
from huggingface_hub import login

HF_TOKEN = ""  # Enter your HuggingFace token
if not HF_TOKEN:
    HF_TOKEN = input("Enter Hugging Face token: ")
login(token=HF_TOKEN)
print("HuggingFace authenticated")

HuggingFace authenticated


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---

## Section 3: Configuration

In [ ]:
@dataclass
class Config:
    # Model
    model_id: str = "meta-llama/Llama-3.2-3B-Instruct"

    # LoRA hyperparameters
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # Training hyperparameters
    batch_size: int = 4
    gradient_accumulation: int = 4
    learning_rate: float = 2e-4
    num_epochs: int = 3
    warmup_ratio: float = 0.1

    # Sequence lengths
    max_input_length: int = 512
    max_output_length: int = 128

    # Paths
    data_dir: str = "/content/drive/MyDrive/dataset_nlp"
    output_dir: str = "/content/outputs"


config = Config()
os.makedirs(config.output_dir, exist_ok=True)

print(f"Model: {config.model_id}")
print(f"Data directory: {config.data_dir}")
print(f"Output directory: {config.output_dir}")

Model: meta-llama/Llama-3.2-3B-Instruct
Data directory: /content/drive/MyDrive/dataset_nlp
Output directory: /content/outputs


---

## Section 4: Data Loading

In [ ]:
def load_jsonl(filepath: str) -> List[Dict]:
    """Load JSONL file."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def standardize(data: List[Dict]) -> List[Dict]:
    """
    Standardize data format.

    CS-SUM uses 'messages' field containing list of message dicts.
    This function extracts conversation text and summary.
    """
    result = []
    for i, item in enumerate(data):
        # Handle CS-SUM messages list format
        if 'messages' in item:
            messages = item['messages']
            conv = '\n'.join([msg.get('text', '') for msg in messages])
        else:
            # Fallback for other formats
            conv = (item.get('conversation') or
                    item.get('dialogue') or
                    item.get('input') or
                    item.get('text') or '')

        summ = item.get('summary') or item.get('output', '')

        if conv and summ:
            result.append({
                'id': item.get('thread_id') or item.get('id', f'item_{i}'),
                'conversation': conv,
                'summary': summ
            })
    return result

In [ ]:
# Load CS-SUM dataset
cssum_dir = os.path.join(config.data_dir, 'cs_sum')

raw_train = load_jsonl(os.path.join(cssum_dir, 'cs_sum_train.jsonl'))
raw_dev = load_jsonl(os.path.join(cssum_dir, 'cs_sum_dev.jsonl'))
raw_test = load_jsonl(os.path.join(cssum_dir, 'cs_sum_test.jsonl'))

train_data = standardize(raw_train)
dev_data = standardize(raw_dev)
test_data = standardize(raw_test)

print(f"CS-SUM dataset loaded:")
print(f"  Train: {len(train_data)} samples")
print(f"  Dev: {len(dev_data)} samples")
print(f"  Test: {len(test_data)} samples")

CS-SUM dataset loaded:
  Train: 2584 samples
  Dev: 323 samples
  Test: 325 samples


In [ ]:
# Data verification
print("=" * 60)
print("DATA VERIFICATION")
print("=" * 60)

print(f"\nFirst conversation (first 500 chars):")
print(train_data[0]['conversation'][:500])
print(f"\nSummary: {train_data[0]['summary']}")

# Sanity checks
assert len(train_data[0]['conversation']) > 50, "ERROR: Conversation too short"
assert len(train_data[0]['summary']) > 10, "ERROR: Summary too short"

print("\nData verification PASSED")

DATA VERIFICATION

First conversation (first 500 chars):
你是不是需要 help with something?
我不知道要去哪里 to get my ballot.
我可以帮你.
你可以怎样帮我?
我在这里工作.
That's great.
我可不可以看 your ID 吗?
Here it is.
All right, 这是你的 ballot card.
我现在应该做什么?
前往 voting booth and 投票.
All right. 感谢您的帮助.

Summary: #Person1# helps #Person2# get a ballot card and guides #Person2# the next step.

Data verification PASSED


In [ ]:
# Load GupShup dataset (Hindi-English for zero-shot evaluation)
gupshup_dir = os.path.join(config.data_dir, 'gupshup')

with open(os.path.join(gupshup_dir, 'test.source'), 'r', encoding='utf-8') as f:
    gupshup_sources = [line.strip() for line in f.readlines()]

with open(os.path.join(gupshup_dir, 'test.target'), 'r', encoding='utf-8') as f:
    gupshup_targets = [line.strip() for line in f.readlines()]

gupshup_data = [
    {'id': f'gupshup_{i}', 'conversation': src, 'summary': tgt}
    for i, (src, tgt) in enumerate(zip(gupshup_sources, gupshup_targets))
]

print(f"GupShup dataset loaded:")
print(f"  Test: {len(gupshup_data)} samples")

GupShup dataset loaded:
  Test: 500 samples


---

## Section 5: Code-Mixing Index (CMI) Computation

The Code-Mixing Index quantifies the degree of language mixing in text. We use script-based detection to identify non-Latin characters (Chinese, Devanagari, Tamil, Arabic).

CMI = 100 * (1 - max_language_words / total_words)

A CMI of 0 means monolingual text, while higher values indicate more code-switching.

In [ ]:
def compute_cmi(text: str) -> float:
    """
    Compute Code-Mixing Index based on script detection.

    Args:
        text: Input text (potentially code-mixed)

    Returns:
        CMI score (0-100). Higher = more code-mixing.
    """
    if not text:
        return 0.0

    words = text.split()
    if len(words) == 0:
        return 0.0

    lang_counts = {'non_latin': 0, 'latin': 0}

    for word in words:
        has_non_latin = False
        for char in word:
            code = ord(char)
            # Check for non-Latin scripts
            if (0x4E00 <= code <= 0x9FFF or    # Chinese
                0x0900 <= code <= 0x097F or    # Devanagari (Hindi)
                0x0B80 <= code <= 0x0BFF or    # Tamil
                0x0600 <= code <= 0x06FF):     # Arabic
                has_non_latin = True
                break

        if has_non_latin:
            lang_counts['non_latin'] += 1
        else:
            lang_counts['latin'] += 1

    total = sum(lang_counts.values())
    if total == 0:
        return 0.0

    max_lang = max(lang_counts.values())
    cmi = 100 * (1 - max_lang / total)
    return cmi

In [ ]:
# Compute CMI distribution for training data
train_cmi = [compute_cmi(x['conversation']) for x in train_data]

print("Training Data CMI Statistics:")
print(f"  Mean: {np.mean(train_cmi):.2f}")
print(f"  Std: {np.std(train_cmi):.2f}")
print(f"  Min: {np.min(train_cmi):.2f}")
print(f"  Max: {np.max(train_cmi):.2f}")
print(f"  Median: {np.median(train_cmi):.2f}")

Training Data CMI Statistics:
  Mean: 11.36
  Std: 16.27
  Min: 0.00
  Max: 50.00
  Median: 0.00


---

## Section 6: Model Setup

In [ ]:
print(f"Loading model: {config.model_id}")

tokenizer = AutoTokenizer.from_pretrained(config.model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
)

model.gradient_checkpointing_enable()
model.config.use_cache = False

print(f"Model loaded successfully")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Layers: {model.config.num_hidden_layers}")

Loading model: meta-llama/Llama-3.2-3B-Instruct


tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded successfully
  Hidden size: 3072
  Layers: 28


In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=config.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"LoRA applied")
print(f"  Trainable parameters: {trainable:,}")
print(f"  Total parameters: {total:,}")
print(f"  Trainable %: {100 * trainable / total:.2f}%")

LoRA applied
  Trainable parameters: 24,313,856
  Total parameters: 3,237,063,680
  Trainable %: 0.75%


---

## Section 7: Dataset and Training Functions

In [ ]:
PROMPT_TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a helpful assistant that summarizes conversations. Provide a clear, concise English summary.<|eot_id|><|start_header_id|>user<|end_header_id|>
Summarize this conversation:

{conversation}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""


class SummarizationDataset(Dataset):
    """Dataset for conversation summarization."""

    def __init__(self, data: List[Dict], tokenizer, max_input_len: int,
                 max_output_len: int, is_train: bool = True):
        self.data = data
        self.tokenizer = tokenizer
        self.max_input_len = max_input_len
        self.max_output_len = max_output_len
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict:
        item = self.data[idx]
        prompt = PROMPT_TEMPLATE.format(conversation=item['conversation'][:1500])

        if self.is_train:
            # For training: include target in sequence
            full_text = prompt + item['summary'] + self.tokenizer.eos_token
            encoding = self.tokenizer(
                full_text,
                truncation=True,
                max_length=self.max_input_len + self.max_output_len,
                padding=False,
                return_tensors=None
            )

            # Mask prompt tokens in labels (only compute loss on output)
            prompt_encoding = self.tokenizer(
                prompt,
                truncation=True,
                max_length=self.max_input_len
            )
            prompt_len = len(prompt_encoding['input_ids'])
            labels = [-100] * prompt_len + encoding['input_ids'][prompt_len:]

            return {
                'input_ids': encoding['input_ids'],
                'attention_mask': encoding['attention_mask'],
                'labels': labels
            }
        else:
            # For inference: only encode prompt
            encoding = self.tokenizer(
                prompt,
                truncation=True,
                max_length=self.max_input_len,
                padding=False,
                return_tensors=None
            )
            return encoding

In [ ]:
def train_model(model, tokenizer, train_data: List[Dict], dev_data: List[Dict],
                config: Config, use_curriculum: bool = True):
    """
    Train model with optional curriculum learning.

    Args:
        model: The model to train
        tokenizer: Tokenizer
        train_data: Training samples
        dev_data: Validation samples
        config: Training configuration
        use_curriculum: If True, use CMI-based curriculum learning

    Returns:
        Trained model
    """
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        padding=True,
        pad_to_multiple_of=8
    )

    if use_curriculum:
        print("\nCURRICULUM LEARNING: 3 PHASES")
        print("=" * 60)

        # Sort training data by CMI (easy to hard)
        data_with_cmi = [(x, compute_cmi(x['conversation'])) for x in train_data]
        data_with_cmi.sort(key=lambda x: x[1])
        sorted_data = [x[0] for x in data_with_cmi]

        # Split into 3 phases
        n = len(sorted_data)
        phases = [
            ('Easy (Low CMI)', sorted_data[:n//3]),
            ('Medium (Mid CMI)', sorted_data[n//3:2*n//3]),
            ('Hard (High CMI)', sorted_data[2*n//3:])
        ]

        for phase_idx, (phase_name, phase_data) in enumerate(phases):
            print(f"\n--- Phase {phase_idx + 1}/3: {phase_name} ---")
            print(f"Training samples: {len(phase_data)}")

            train_dataset = SummarizationDataset(
                phase_data, tokenizer,
                config.max_input_length, config.max_output_length,
                is_train=True
            )

            dev_dataset = SummarizationDataset(
                dev_data, tokenizer,
                config.max_input_length, config.max_output_length,
                is_train=True
            )

            training_args = TrainingArguments(
                output_dir=f"{config.output_dir}/phase_{phase_idx}",
                num_train_epochs=1,
                per_device_train_batch_size=config.batch_size,
                per_device_eval_batch_size=config.batch_size,
                gradient_accumulation_steps=config.gradient_accumulation,
                learning_rate=config.learning_rate,
                warmup_ratio=config.warmup_ratio,
                logging_steps=50,
                eval_strategy="steps",
                eval_steps=50,
                save_strategy="no",
                bf16=True,
                report_to="none",
                remove_unused_columns=False
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=dev_dataset,
                data_collator=data_collator
            )

            trainer.train()
            print(f"Phase {phase_idx + 1} complete")

    else:
        print("\nSTANDARD TRAINING (No Curriculum)")
        print("=" * 60)

        train_dataset = SummarizationDataset(
            train_data, tokenizer,
            config.max_input_length, config.max_output_length,
            is_train=True
        )

        dev_dataset = SummarizationDataset(
            dev_data, tokenizer,
            config.max_input_length, config.max_output_length,
            is_train=True
        )

        training_args = TrainingArguments(
            output_dir=f"{config.output_dir}/standard",
            num_train_epochs=config.num_epochs,
            per_device_train_batch_size=config.batch_size,
            per_device_eval_batch_size=config.batch_size,
            gradient_accumulation_steps=config.gradient_accumulation,
            learning_rate=config.learning_rate,
            warmup_ratio=config.warmup_ratio,
            logging_steps=50,
            eval_strategy="steps",
            eval_steps=50,
            save_strategy="no",
            bf16=True,
            report_to="none",
            remove_unused_columns=False
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=dev_dataset,
            data_collator=data_collator
        )

        trainer.train()

    print("\nTraining complete")
    return model

---

## Section 8: Train CSCL Model

In [ ]:
print("Training with Code-Switching Curriculum Learning (CSCL)...")
model = train_model(model, tokenizer, train_data, dev_data, config, use_curriculum=True)

Training with Code-Switching Curriculum Learning (CSCL)...

CURRICULUM LEARNING: 3 PHASES

--- Phase 1/3: Easy (Low CMI) ---
Training samples: 861


Step,Training Loss,Validation Loss
50,1.549200,1.419806


Phase 1 complete

--- Phase 2/3: Medium (Mid CMI) ---
Training samples: 861


Step,Training Loss,Validation Loss
50,1.584600,1.330375


Phase 2 complete

--- Phase 3/3: Hard (High CMI) ---
Training samples: 862


Step,Training Loss,Validation Loss
50,1.247400,1.274261


Phase 3 complete

Training complete


---

## Section 9: Evaluation Functions

In [ ]:
# ============================================================
# CELL 28 - FIXED VERSION (escaped braces)
# Replace your current Cell 28 with this code
# ============================================================

rouge_metric = evaluate.load("rouge")


def compute_all_metrics(predictions: List[str], references: List[str]) -> Dict:
    """
    Compute ROUGE and BERTScore metrics.
    """
    from bert_score import score as bert_score

    # Filter empty predictions
    valid = [(p, r) for p, r in zip(predictions, references) if p.strip()]
    if not valid:
        return {'rouge1': 0, 'rouge2': 0, 'rougeL': 0, 'bertscore_f1': 0}

    preds, refs = zip(*valid)
    preds, refs = list(preds), list(refs)

    # ROUGE scores
    rouge_results = rouge_metric.compute(predictions=preds, references=refs)

    # BERTScore
    P, R, F1 = bert_score(preds, refs, lang='en', verbose=False)

    return {
        'rouge1': rouge_results['rouge1'],
        'rouge2': rouge_results['rouge2'],
        'rougeL': rouge_results['rougeL'],
        'bertscore_f1': F1.mean().item()
    }


def clean_summary(text: str) -> str:
    """
    Clean up generated summary text.
    """
    if not text:
        return ""

    text = str(text).strip()

    # Remove 'assistant' prefix
    if text.lower().startswith('assistant'):
        text = text[9:].strip()

    # Remove leading newlines
    text = text.lstrip('\n').strip()

    # Remove special tokens
    for token in ['<|eot_id|>', '<|end|>', '</s>', '<|im_end|>', '<|assistant|>']:
        text = text.replace(token, '')

    # Clean up extra whitespace
    text = ' '.join(text.split())

    return text.strip()


def generate_summaries(model, tokenizer, data: List[Dict],
                       batch_size: int = 4) -> List[str]:
    """
    Generate high-quality summaries for a dataset.
    """
    model.eval()
    predictions = []

    # CRITICAL: Set left padding for decoder-only models
    original_padding_side = tokenizer.padding_side
    tokenizer.padding_side = 'left'

    for i in range(0, len(data), batch_size):
        batch = data[i:i + batch_size]

        # Use the original PROMPT_TEMPLATE (defined elsewhere in notebook)
        prompts = [PROMPT_TEMPLATE.format(conversation=x['conversation'][:2000])
                   for x in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=600
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                min_new_tokens=10,
                num_beams=5,
                do_sample=False,
                repetition_penalty=1.2,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        for j, output in enumerate(outputs):
            input_len = inputs['input_ids'][j].shape[0]
            generated_tokens = output[input_len:]
            generated = tokenizer.decode(generated_tokens, skip_special_tokens=True)
            cleaned = clean_summary(generated)
            predictions.append(cleaned)

        if (i + batch_size) % 40 == 0:
            print(f"  Generated {min(i + batch_size, len(data))}/{len(data)}")

    tokenizer.padding_side = original_padding_side
    return predictions

---

## Section 10: Base Llama Baseline

Evaluate the base Llama model (without fine-tuning) to establish a baseline.

In [ ]:
print("Loading base Llama model (no fine-tuning) for baseline...")

base_model = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

base_tokenizer = AutoTokenizer.from_pretrained(config.model_id)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "right"

print("Base model loaded")

Loading base Llama model (no fine-tuning) for baseline...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Base model loaded


In [ ]:
# Evaluate base Llama on CS-SUM (FULL test set)
print(f"Evaluating base Llama on CS-SUM ({len(test_data)} samples)...")
base_cssum_preds = generate_summaries(base_model, base_tokenizer, test_data)
base_cssum_metrics = compute_all_metrics(base_cssum_preds,
                                          [x['summary'] for x in test_data])
print(f"Base Llama CS-SUM ROUGE-L: {base_cssum_metrics['rougeL']:.4f}")

Evaluating base Llama on CS-SUM (325 samples)...
  Generated 40/325
  Generated 80/325
  Generated 120/325
  Generated 160/325
  Generated 200/325
  Generated 240/325
  Generated 280/325
  Generated 320/325


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base Llama CS-SUM ROUGE-L: 0.1575


In [ ]:
# Evaluate base Llama on GupShup (FULL test set - zero-shot)
print(f"Evaluating base Llama on GupShup ({len(gupshup_data)} samples)...")
base_gupshup_preds = generate_summaries(base_model, base_tokenizer, gupshup_data)
base_gupshup_metrics = compute_all_metrics(base_gupshup_preds,
                                            [x['summary'] for x in gupshup_data])
print(f"Base Llama GupShup ROUGE-L: {base_gupshup_metrics['rougeL']:.4f}")

# Clean up base model to free GPU memory
del base_model
torch.cuda.empty_cache()
gc.collect()

Evaluating base Llama on GupShup (500 samples)...
  Generated 40/500
  Generated 80/500
  Generated 120/500
  Generated 160/500
  Generated 200/500
  Generated 240/500
  Generated 280/500
  Generated 320/500
  Generated 360/500
  Generated 400/500
  Generated 440/500
  Generated 480/500


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base Llama GupShup ROUGE-L: 0.1877


0

---

## Section 11: Evaluate Fine-tuned CSCL Model

In [ ]:
print("=" * 60)
print("EVALUATING FINE-TUNED MODEL (CSCL)")
print("=" * 60)

# CS-SUM (in-domain) - FULL TEST SET
print(f"\nCS-SUM (in-domain) - {len(test_data)} samples...")
cssum_preds = generate_summaries(model, tokenizer, test_data)
cssum_refs = [x['summary'] for x in test_data]
cssum_metrics = compute_all_metrics(cssum_preds, cssum_refs)

print(f"  ROUGE-1: {cssum_metrics['rouge1']:.4f}")
print(f"  ROUGE-2: {cssum_metrics['rouge2']:.4f}")
print(f"  ROUGE-L: {cssum_metrics['rougeL']:.4f}")
print(f"  BERTScore F1: {cssum_metrics['bertscore_f1']:.4f}")

EVALUATING FINE-TUNED MODEL (CSCL)

CS-SUM (in-domain) - 325 samples...
  Generated 40/325
  Generated 80/325
  Generated 120/325
  Generated 160/325
  Generated 200/325
  Generated 240/325
  Generated 280/325
  Generated 320/325


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ROUGE-1: 0.4154
  ROUGE-2: 0.1669
  ROUGE-L: 0.3399
  BERTScore F1: 0.9109


In [ ]:
# GupShup (zero-shot Hindi-English) - FULL TEST SET
print(f"\nGupShup (zero-shot Hindi-English) - {len(gupshup_data)} samples...")
gupshup_preds = generate_summaries(model, tokenizer, gupshup_data)
gupshup_refs = [x['summary'] for x in gupshup_data]
gupshup_metrics = compute_all_metrics(gupshup_preds, gupshup_refs)

print(f"  ROUGE-1: {gupshup_metrics['rouge1']:.4f}")
print(f"  ROUGE-2: {gupshup_metrics['rouge2']:.4f}")
print(f"  ROUGE-L: {gupshup_metrics['rougeL']:.4f}")
print(f"  BERTScore F1: {gupshup_metrics['bertscore_f1']:.4f}")


GupShup (zero-shot Hindi-English) - 500 samples...
  Generated 40/500
  Generated 80/500
  Generated 120/500
  Generated 160/500
  Generated 200/500
  Generated 240/500
  Generated 280/500
  Generated 320/500
  Generated 360/500
  Generated 400/500
  Generated 440/500
  Generated 480/500


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ROUGE-1: 0.4485
  ROUGE-2: 0.1901
  ROUGE-L: 0.3608
  BERTScore F1: 0.9112


In [ ]:
# Sample predictions
print("\nSample Predictions (GupShup - Zero-shot):")
print("-" * 60)

for i in range(3):
    print(f"\nExample {i + 1}:")
    print(f"Conversation: {gupshup_data[i]['conversation'][:200]}...")
    print(f"Reference: {gupshup_refs[i]}")
    print(f"Prediction: {gupshup_preds[i]}")


Sample Predictions (GupShup - Zero-shot):
------------------------------------------------------------

Example 1:
Conversation: Ivy: Chloene bataya tum humare saath nahi aa rahe! Carter: mera ek family reunion around that time Ivy: just ditch it Carter: nahi kar sakta, iss baar nahi Ivy: why? Carter: mere grandfather kee tabiy...
Reference: Carter is not joining Ivy and Chloe due to a family reunion. Carter's grandfather is very ill.
Prediction: Carter's grandfather's health is bad, so he can't go to the family reunion. Carter will go with Ivy on the next trip.

Example 2:
Conversation: Ingrid: <file_photo> <file_photo> <file_photo> Ingrid: Tree's up!! Bart: Looking good! Baubles bahut paas paas nahi hai? Ingrid: Sammie's fine work! voh unko akela nahi chodna chahti thi... 😂 😍 Bart: ...
Reference: Sammie's put the baubles very close to each other on the tree and Ingrid decides to leave it like that for some time and then discuss it with Sammie. Noah prepared a creative version of the

---

## Section 12: Ablation - Training Without Curriculum

To understand the effect of curriculum learning, we train an identical model without the CMI-based ordering (standard random batching for 3 epochs).

In [ ]:
print("=" * 60)
print("ABLATION: Training WITHOUT Curriculum")
print("=" * 60)

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

# Reload fresh model for ablation
model_ablation = AutoModelForCausalLM.from_pretrained(
    config.model_id,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager"
).to("cuda")

model_ablation.gradient_checkpointing_enable()
model_ablation.config.use_cache = False
model_ablation = get_peft_model(model_ablation, lora_config)

print("Training without curriculum (standard 3 epochs)...")
model_ablation = train_model(model_ablation, tokenizer, train_data, dev_data,
                              config, use_curriculum=False)

ABLATION: Training WITHOUT Curriculum


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Training without curriculum (standard 3 epochs)...

STANDARD TRAINING (No Curriculum)


Step,Training Loss,Validation Loss
50,1.878300,1.430921
100,1.381200,1.309202
150,1.241000,1.207776
200,0.920100,1.167024
250,0.838600,1.068766
300,0.723100,0.989028
350,0.504900,1.004826
400,0.337400,0.982369
450,0.306300,0.944332



Training complete


In [ ]:
# Evaluate ablation model on FULL test sets
print("\nEvaluating ablation model (No Curriculum)...")

print(f"CS-SUM - {len(test_data)} samples...")
ablation_cssum_preds = generate_summaries(model_ablation, tokenizer, test_data)
ablation_cssum_metrics = compute_all_metrics(ablation_cssum_preds, cssum_refs)

print(f"GupShup - {len(gupshup_data)} samples...")
ablation_gupshup_preds = generate_summaries(model_ablation, tokenizer, gupshup_data)
ablation_gupshup_metrics = compute_all_metrics(ablation_gupshup_preds, gupshup_refs)

print(f"\nNo Curriculum Results:")
print(f"  CS-SUM ROUGE-L: {ablation_cssum_metrics['rougeL']:.4f}")
print(f"  GupShup ROUGE-L: {ablation_gupshup_metrics['rougeL']:.4f}")

# Clean up
del model_ablation
torch.cuda.empty_cache()
gc.collect()


Evaluating ablation model (No Curriculum)...
CS-SUM - 325 samples...
  Generated 40/325
  Generated 80/325
  Generated 120/325
  Generated 160/325
  Generated 200/325
  Generated 240/325
  Generated 280/325
  Generated 320/325


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


GupShup - 500 samples...
  Generated 40/500
  Generated 80/500
  Generated 120/500
  Generated 160/500
  Generated 200/500
  Generated 240/500
  Generated 280/500
  Generated 320/500
  Generated 360/500
  Generated 400/500
  Generated 440/500
  Generated 480/500


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



No Curriculum Results:
  CS-SUM ROUGE-L: 0.4151
  GupShup ROUGE-L: 0.3234


35536

---

## Section 13: GPT-4.1 Baseline (Fair Comparison)

We compare against GPT-4.1 using a fair evaluation methodology:

1. **Temperature = 0**: Deterministic output for reproducibility
2. **Same prompt style**: No artificial length constraints
3. **Full test set**: Same samples as our fine-tuned model

In [ ]:
from openai import OpenAI
import time

OPENAI_API_KEY = ""  # Enter your OpenAI API key
if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("Enter OpenAI API key: ")

client = OpenAI(api_key=OPENAI_API_KEY)


def get_gpt_summary(conversation: str, max_retries: int = 3) -> str:
    """
    Get summary from GPT-4.1 with fair comparison settings.

    Fair comparison methodology:
    - temperature=0: Deterministic output for reproducibility
    - Same prompt style as Llama: No artificial length constraints
    - max_tokens=150: Allow outputs matching reference length distribution

    Args:
        conversation: The conversation to summarize
        max_retries: Number of retry attempts on failure

    Returns:
        Generated summary string
    """
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4.1",
                messages=[
                    {
                        "role": "system",
                        "content": "You are a helpful assistant that summarizes conversations. Provide a clear, concise English summary."
                    },
                    {
                        "role": "user",
                        "content": f"Summarize this conversation:\n\n{conversation}"
                    }
                ],
                max_tokens=150,
                temperature=0  # Deterministic for reproducibility
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                return ""
    return ""

In [ ]:
# Evaluate GPT-4.1 on FULL CS-SUM test set
print(f"GPT-4.1 on CS-SUM (FULL test set: {len(test_data)} samples)...")
print("Settings: temperature=0, no length constraint")
print("-" * 60)

gpt_cssum_preds = []
for i, item in enumerate(test_data):
    pred = get_gpt_summary(item['conversation'])
    gpt_cssum_preds.append(pred)
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(test_data)}")

gpt_cssum_metrics = compute_all_metrics(gpt_cssum_preds, cssum_refs)
print(f"\nGPT-4.1 CS-SUM ROUGE-L: {gpt_cssum_metrics['rougeL']:.4f}")

GPT-4.1 on CS-SUM (FULL test set: 325 samples)...
Settings: temperature=0, no length constraint
------------------------------------------------------------
  50/325
  100/325
  150/325
  200/325
  250/325
  300/325


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



GPT-4.1 CS-SUM ROUGE-L: 0.2028


In [ ]:
# Evaluate GPT-4.1 on FULL GupShup test set
print(f"\nGPT-4.1 on GupShup (FULL test set: {len(gupshup_data)} samples)...")
print("Settings: temperature=0, no length constraint")
print("-" * 60)

gpt_gupshup_preds = []
for i, item in enumerate(gupshup_data):
    pred = get_gpt_summary(item['conversation'])
    gpt_gupshup_preds.append(pred)
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{len(gupshup_data)}")

gpt_gupshup_metrics = compute_all_metrics(gpt_gupshup_preds, gupshup_refs)
print(f"\nGPT-4.1 GupShup ROUGE-L: {gpt_gupshup_metrics['rougeL']:.4f}")


GPT-4.1 on GupShup (FULL test set: 500 samples)...
Settings: temperature=0, no length constraint
------------------------------------------------------------
  50/500
  100/500
  150/500
  200/500
  250/500
  300/500
  350/500
  400/500
  450/500
  500/500


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



GPT-4.1 GupShup ROUGE-L: 0.2407


In [ ]:
# Sample GPT-4.1 predictions
print("\nSample GPT-4.1 Predictions (CS-SUM):")
print("-" * 60)

for i in range(3):
    print(f"\nExample {i + 1}:")
    print(f"Reference: {cssum_refs[i]}")
    print(f"GPT-4.1: {gpt_cssum_preds[i]}")


Sample GPT-4.1 Predictions (CS-SUM):
------------------------------------------------------------

Example 1:
Reference: Callum is still busy.
GPT-4.1: Jair asked Callum if he was still busy. Callum replied that he was still a little busy and apologized. Jair acknowledged his response.

Example 2:
Reference: Daniel, Michael, Matt and Brian are going to Croatia and Bosnia and Herzegovina. They are packing. Michael reminds them to take their passports, because Bosnia and Herzegovina is not in the EU. They will go to Mostar and the mountains in Bosnia.
GPT-4.1: The group is preparing for a trip. Daniel asks if everyone has finished packing, and Michael reminds them not to forget their passports. Matt asks if an ID is enough, but Michael explains that since Bosnia and Herzegovina is not in the EU, they will face proper border control and need passports to enter. Brian was unaware they would be entering Bosnia, thinking they would only stay in Croatia, but Michael clarifies that their plan

---

## Section 14: Final Results Comparison

In [ ]:
print("=" * 80)
print("FINAL RESULTS")
print("=" * 80)

print(f"\n{'Model':<30} {'CS-SUM':<15} {'GupShup':<15} {'Samples':<15} {'Notes'}")
print("-" * 90)
print(f"{'Base Llama (no fine-tuning)':<30} {base_cssum_metrics['rougeL']:<15.4f} {base_gupshup_metrics['rougeL']:<15.4f} {'Full test':<15} {'Pretrained only'}")
print(f"{'GPT-4.1':<30} {gpt_cssum_metrics['rougeL']:<15.4f} {gpt_gupshup_metrics['rougeL']:<15.4f} {'Full test':<15} {'temp=0, fair prompt'}")
print(f"{'No Curriculum (3 epochs)':<30} {ablation_cssum_metrics['rougeL']:<15.4f} {ablation_gupshup_metrics['rougeL']:<15.4f} {'Full test':<15} {'Standard fine-tuning'}")
print(f"{'CSCL (Ours)':<30} {cssum_metrics['rougeL']:<15.4f} {gupshup_metrics['rougeL']:<15.4f} {'Full test':<15} {'Curriculum learning'}")

FINAL RESULTS

Model                          CS-SUM          GupShup         Samples         Notes
------------------------------------------------------------------------------------------
Base Llama (no fine-tuning)    0.1575          0.1877          Full test       Pretrained only
GPT-4.1                        0.2028          0.2407          Full test       temp=0, fair prompt
No Curriculum (3 epochs)       0.4151          0.3234          Full test       Standard fine-tuning
CSCL (Ours)                    0.3399          0.3608          Full test       Curriculum learning


In [ ]:
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)

# 1. Improvement over base Llama
cssum_improvement = (cssum_metrics['rougeL'] - base_cssum_metrics['rougeL']) / base_cssum_metrics['rougeL'] * 100
gupshup_improvement = (gupshup_metrics['rougeL'] - base_gupshup_metrics['rougeL']) / base_gupshup_metrics['rougeL'] * 100

print(f"\n1. Fine-tuning improves over base Llama:")
print(f"   CS-SUM: {base_cssum_metrics['rougeL']:.4f} -> {cssum_metrics['rougeL']:.4f} (+{cssum_improvement:.1f}%)")
print(f"   GupShup: {base_gupshup_metrics['rougeL']:.4f} -> {gupshup_metrics['rougeL']:.4f} (+{gupshup_improvement:.1f}%)")

# 2. Comparison with GPT-4.1
cssum_vs_gpt = (cssum_metrics['rougeL'] - gpt_cssum_metrics['rougeL']) / gpt_cssum_metrics['rougeL'] * 100
gupshup_vs_gpt = (gupshup_metrics['rougeL'] - gpt_gupshup_metrics['rougeL']) / gpt_gupshup_metrics['rougeL'] * 100

print(f"\n2. Comparison with GPT-4.1 (fair evaluation):")
print(f"   CS-SUM: CSCL {cssum_metrics['rougeL']:.4f} vs GPT-4.1 {gpt_cssum_metrics['rougeL']:.4f} ({'+' if cssum_vs_gpt > 0 else ''}{cssum_vs_gpt:.1f}%)")
print(f"   GupShup: CSCL {gupshup_metrics['rougeL']:.4f} vs GPT-4.1 {gpt_gupshup_metrics['rougeL']:.4f} ({'+' if gupshup_vs_gpt > 0 else ''}{gupshup_vs_gpt:.1f}%)")

# 3. Zero-shot transfer
print(f"\n3. Zero-shot transfer:")
print(f"   Trained on: Chinese-English (CS-SUM)")
print(f"   Tested on: Hindi-English (GupShup) - NEVER SEEN IN TRAINING")
print(f"   GupShup ROUGE-L: {gupshup_metrics['rougeL']:.4f} (comparable to in-domain {cssum_metrics['rougeL']:.4f})")

# 4. Curriculum learning effect
print(f"\n4. Curriculum learning effect:")
print(f"   In-domain (CS-SUM): CSCL {cssum_metrics['rougeL']:.4f} vs No-Curr {ablation_cssum_metrics['rougeL']:.4f}")
print(f"   Zero-shot (GupShup): CSCL {gupshup_metrics['rougeL']:.4f} vs No-Curr {ablation_gupshup_metrics['rougeL']:.4f}")
print(f"   -> Curriculum trades in-domain performance for better cross-lingual generalization")


KEY FINDINGS

1. Fine-tuning improves over base Llama:
   CS-SUM: 0.1575 -> 0.3399 (+115.8%)
   GupShup: 0.1877 -> 0.3608 (+92.2%)

2. Comparison with GPT-4.1 (fair evaluation):
   CS-SUM: CSCL 0.3399 vs GPT-4.1 0.2028 (+67.6%)
   GupShup: CSCL 0.3608 vs GPT-4.1 0.2407 (+49.9%)

3. Zero-shot transfer:
   Trained on: Chinese-English (CS-SUM)
   Tested on: Hindi-English (GupShup) - NEVER SEEN IN TRAINING
   GupShup ROUGE-L: 0.3608 (comparable to in-domain 0.3399)

4. Curriculum learning effect:
   In-domain (CS-SUM): CSCL 0.3399 vs No-Curr 0.4151
   Zero-shot (GupShup): CSCL 0.3608 vs No-Curr 0.3234
   -> Curriculum trades in-domain performance for better cross-lingual generalization


---

## Section 15: Data Leakage Check

Verify that results are not due to memorization or data leakage.

In [ ]:
print("=" * 60)
print("DATA LEAKAGE CHECK")
print("=" * 60)

# Check for exact matches between predictions and references
exact_matches = sum(
    1 for p, r in zip(cssum_preds, cssum_refs)
    if p.strip().lower() == r.strip().lower()
)
print(f"\nExact matches: {exact_matches}/{len(cssum_preds)} ({100 * exact_matches / len(cssum_preds):.1f}%)")

# Check for high word overlap (>90%)
high_overlap = 0
for pred, ref in zip(cssum_preds, cssum_refs):
    pred_words = set(pred.lower().split())
    ref_words = set(ref.lower().split())
    if len(ref_words) > 0:
        overlap = len(pred_words & ref_words) / len(ref_words)
        if overlap > 0.9:
            high_overlap += 1

print(f"High word overlap (>90%): {high_overlap}/{len(cssum_preds)} ({100 * high_overlap / len(cssum_preds):.1f}%)")

# Check train-test conversation overlap
train_convs = set(x['conversation'][:100] for x in train_data)
test_convs = set(x['conversation'][:100] for x in test_data)
overlap_count = len(train_convs & test_convs)
print(f"Train-test conversation overlap: {overlap_count}")

# Verdict
if exact_matches < 5 and high_overlap < 10 and overlap_count == 0:
    print("\nVerdict: No data leakage detected")
else:
    print("\nWarning: Potential data leakage detected - investigate further")

DATA LEAKAGE CHECK

Exact matches: 0/325 (0.0%)
High word overlap (>90%): 0/325 (0.0%)
Train-test conversation overlap: 0

Verdict: No data leakage detected


---

## Section 16: Save Results

In [ ]:
# Save comparison table
results = [
    {
        'Model': 'Base Llama',
        'CS-SUM ROUGE-L': base_cssum_metrics['rougeL'],
        'GupShup ROUGE-L': base_gupshup_metrics['rougeL'],
        'CS-SUM BERTScore': base_cssum_metrics['bertscore_f1'],
        'GupShup BERTScore': base_gupshup_metrics['bertscore_f1'],
        'CS-SUM Samples': len(test_data),
        'GupShup Samples': len(gupshup_data),
        'Notes': 'Pretrained only'
    },
    {
        'Model': 'GPT-4.1',
        'CS-SUM ROUGE-L': gpt_cssum_metrics['rougeL'],
        'GupShup ROUGE-L': gpt_gupshup_metrics['rougeL'],
        'CS-SUM BERTScore': gpt_cssum_metrics['bertscore_f1'],
        'GupShup BERTScore': gpt_gupshup_metrics['bertscore_f1'],
        'CS-SUM Samples': len(test_data),
        'GupShup Samples': len(gupshup_data),
        'Notes': 'temp=0, fair prompt'
    },
    {
        'Model': 'No Curriculum',
        'CS-SUM ROUGE-L': ablation_cssum_metrics['rougeL'],
        'GupShup ROUGE-L': ablation_gupshup_metrics['rougeL'],
        'CS-SUM BERTScore': ablation_cssum_metrics['bertscore_f1'],
        'GupShup BERTScore': ablation_gupshup_metrics['bertscore_f1'],
        'CS-SUM Samples': len(test_data),
        'GupShup Samples': len(gupshup_data),
        'Notes': 'Standard fine-tuning (3 epochs)'
    },
    {
        'Model': 'CSCL (Ours)',
        'CS-SUM ROUGE-L': cssum_metrics['rougeL'],
        'GupShup ROUGE-L': gupshup_metrics['rougeL'],
        'CS-SUM BERTScore': cssum_metrics['bertscore_f1'],
        'GupShup BERTScore': gupshup_metrics['bertscore_f1'],
        'CS-SUM Samples': len(test_data),
        'GupShup Samples': len(gupshup_data),
        'Notes': 'Curriculum learning'
    },
]

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(config.output_dir, 'final_results.csv'), index=False)

print("Saved: final_results.csv")
print("\n" + results_df.to_string(index=False))

Saved: final_results.csv

        Model  CS-SUM ROUGE-L  GupShup ROUGE-L  CS-SUM BERTScore  GupShup BERTScore  CS-SUM Samples  GupShup Samples                           Notes
   Base Llama        0.157466         0.187714          0.862405           0.869669             325              500                 Pretrained only
      GPT-4.1        0.202825         0.240660          0.877629           0.885812             325              500             temp=0, fair prompt
No Curriculum        0.415074         0.323350          0.919074           0.906091             325              500 Standard fine-tuning (3 epochs)
  CSCL (Ours)        0.339882         0.360752          0.910908           0.911228             325              500             Curriculum learning


In [ ]:
# Save predictions for all models

# CSCL predictions
cssum_output = pd.DataFrame([
    {'id': test_data[i]['id'], 'reference': cssum_refs[i], 'prediction': cssum_preds[i]}
    for i in range(len(test_data))
])
cssum_output.to_csv(os.path.join(config.output_dir, 'cssum_predictions.csv'), index=False)

gupshup_output = pd.DataFrame([
    {'id': gupshup_data[i]['id'], 'reference': gupshup_refs[i], 'prediction': gupshup_preds[i]}
    for i in range(len(gupshup_data))
])
gupshup_output.to_csv(os.path.join(config.output_dir, 'gupshup_predictions.csv'), index=False)

# GPT-4.1 predictions
gpt_cssum_output = pd.DataFrame([
    {'id': test_data[i]['id'], 'reference': cssum_refs[i], 'prediction': gpt_cssum_preds[i]}
    for i in range(len(test_data))
])
gpt_cssum_output.to_csv(os.path.join(config.output_dir, 'gpt_cssum_predictions.csv'), index=False)

gpt_gupshup_output = pd.DataFrame([
    {'id': gupshup_data[i]['id'], 'reference': gupshup_refs[i], 'prediction': gpt_gupshup_preds[i]}
    for i in range(len(gupshup_data))
])
gpt_gupshup_output.to_csv(os.path.join(config.output_dir, 'gpt_gupshup_predictions.csv'), index=False)

print("Saved prediction files:")
print("  - cssum_predictions.csv")
print("  - gupshup_predictions.csv")
print("  - gpt_cssum_predictions.csv")
print("  - gpt_gupshup_predictions.csv")

Saved prediction files:
  - cssum_predictions.csv
  - gupshup_predictions.csv
  - gpt_cssum_predictions.csv
  - gpt_gupshup_predictions.csv


In [ ]:
# Save trained model
model_path = os.path.join(config.output_dir, 'cscl_model')
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f"Model saved to: {model_path}")

Model saved to: /content/outputs/cscl_model


---

## Summary

This notebook demonstrates Code-Switching Curriculum Learning (CSCL) for cross-lingual conversation summarization.

### Key Results

1. **Fine-tuned 3B Llama outperforms GPT-4.1** on both in-domain (CS-SUM) and zero-shot (GupShup) evaluation

2. **Zero-shot transfer works**: Model trained only on Chinese-English achieves comparable performance on Hindi-English without any Hindi training data

3. **Curriculum learning trade-off**: CMI-based curriculum sacrifices some in-domain performance for improved cross-lingual generalization

### Outputs

- `final_results.csv`: Comparison table with all metrics
- `cssum_predictions.csv`: CSCL model predictions on CS-SUM
- `gupshup_predictions.csv`: CSCL model predictions on GupShup
- `gpt_cssum_predictions.csv`: GPT-4.1 predictions on CS-SUM
- `gpt_gupshup_predictions.csv`: GPT-4.1 predictions on GupShup
- `cscl_model/`: Saved fine-tuned model weights